# Сердечно-сосудистые заболевания

## Загрузка необходимых модулей

In [1]:
import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier, BaggingClassifier
from sklearn.model_selection import GridSearchCV

## Работа с данными

In [2]:
data = pd.read_csv("heart_failure_clinical_records_dataset.csv")

In [3]:
data.head()

,age,anaemia,creatinine_phosphokinase,diabetes,ejection_fraction,high_blood_pressure,platelets,serum_creatinine,serum_sodium,sex,smoking,time,DEATH_EVENT
0,75.0,0,582,0,20,1,265000.00,1.9,130,1,0,4,1
1,55.0,0,7861,0,38,0,263358.03,1.1,136,1,0,6,1
2,65.0,0,146,0,20,0,162000.00,1.3,129,1,1,7,1
3,50.0,1,111,0,20,0,210000.00,1.9,137,1,0,7,1
4,65.0,1,160,1,20,0,327000.00,2.7,116,0,0,8,1


In [4]:
data.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 299 entries, 0 to 298
Data columns (total 13 columns):
 #   Column                    Non-Null Count  Dtype  
---  ------                    --------------  -----  
 0   age                       299 non-null    float64
 1   anaemia                   299 non-null    int64  
 2   creatinine_phosphokinase  299 non-null    int64  
 3   diabetes                  299 non-null    int64  
 4   ejection_fraction         299 non-null    int64  
 5   high_blood_pressure       299 non-null    int64  
 6   platelets                 299 non-null    float64
 7   serum_creatinine          299 non-null    float64
 8   serum_sodium              299 non-null    int64  
 9   sex                       299 non-null    int64  
 10  smoking                   299 non-null    int64  
 11  time                      299 non-null    int64  
 12  DEATH_EVENT               299 non-null    int64  
dtypes: float64(3), int64(10)
memory usage: 30.5 KB


**Видим, что нет пропусков**

In [5]:
data.isnull().sum()

age                         0
anaemia                     0
creatinine_phosphokinase    0
diabetes                    0
ejection_fraction           0
high_blood_pressure         0
platelets                   0
serum_creatinine            0
serum_sodium                0
sex                         0
smoking                     0
time                        0
DEATH_EVENT                 0
dtype: int64

**Видим, что нет null значений**

In [6]:
print(data['DEATH_EVENT'].value_counts())

DEATH_EVENT
0    203
1     96
Name: count, dtype: int64


In [3]:
print(data['DEATH_EVENT'].value_counts(normalize=True) * 100)

DEATH_EVENT
0    67.892977
1    32.107023
Name: proportion, dtype: float64


## Модель

**Разделяем датасет на признаки и целtвой**

In [6]:
X = data.drop('DEATH_EVENT', axis=1)
y = data['DEATH_EVENT']

**Разделяем X и y на трейн и тест**

In [7]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=17, stratify=y
)

### Случайный лес

In [8]:
rf = RandomForestClassifier(random_state=17)

In [9]:
rf.fit(X_train, y_train)

RandomForestClassifier(random_state=17)

In [10]:
y_pred = rf.predict(X_test)

In [11]:
accuracy = accuracy_score(y_test, y_pred)
accuracy

0.8833333333333333

### Кросс-валидируем лес

In [12]:
parameters = {
    "max_features": [1, 2, 4, 'sqrt'],
    "min_samples_leaf": [2, 3, 5, 7, 9],
    "max_depth": [5, 10, 15],
}

In [13]:
grid_search_tree = GridSearchCV(
    estimator=rf,
    param_grid=parameters,
    n_jobs=-1,
)

In [14]:
grid_search_tree.fit(X_train, y_train)

GridSearchCV(estimator=RandomForestClassifier(random_state=17), n_jobs=-1,
             param_grid={'max_depth': [5, 10, 15],
                         'max_features': [1, 2, 4, 'sqrt'],
                         'min_samples_leaf': [2, 3, 5, 7, 9]})

In [15]:
best_tree = RandomForestClassifier(max_depth = 5, max_features = 4, min_samples_leaf = 5, random_state=17)

In [16]:
best_tree.fit(X_train, y_train)

RandomForestClassifier(max_depth=5, max_features=4, min_samples_leaf=5,
                       random_state=17)

In [17]:
best_tree_pred = best_tree.predict(X_test)

In [18]:
accuracy1 = accuracy_score(y_test, best_tree_pred)
accuracy1

0.8833333333333333

**Результат валидированного леса меньше из-за размера датасета - он слишком мал, поэтому ошибка в 1 предсказание даёт 1 процент разницы**

### Бэггинг деревьев

In [19]:
b_tree = BaggingClassifier(DecisionTreeClassifier(), n_estimators = 100, random_state = 17)

In [20]:
b_tree.fit(X_train, y_train)

BaggingClassifier(estimator=DecisionTreeClassifier(), n_estimators=100,
                  random_state=17)

In [21]:
b_pred = b_tree.predict(X_test)

In [22]:
accuracy2 = accuracy_score(y_test, b_pred)
accuracy2

0.8166666666666667

### Кросс валидируем бэггинг

In [23]:
parameters2 = {
    "estimator__max_features": [1, 2, 4, 'sqrt'],
    "estimator__min_samples_leaf": [2, 3, 5, 7, 9],
    "estimator__max_depth": [5, 10, 15],
}

In [24]:
grid_search_bag = GridSearchCV(
    estimator=b_tree,
    param_grid=parameters2,
    n_jobs=-1,
)

In [25]:
grid_search_bag.fit(X_train, y_train)

GridSearchCV(estimator=BaggingClassifier(estimator=DecisionTreeClassifier(),
                                         n_estimators=100, random_state=17),
             n_jobs=-1,
             param_grid={'estimator__max_depth': [5, 10, 15],
                         'estimator__max_features': [1, 2, 4, 'sqrt'],
                         'estimator__min_samples_leaf': [2, 3, 5, 7, 9]})

In [26]:
best_bag = BaggingClassifier(estimator = DecisionTreeClassifier(max_depth = 5, min_samples_leaf = 2, max_features = 4), n_estimators = 100, random_state = 17)

In [27]:
best_bag.fit(X_train, y_train)

BaggingClassifier(estimator=DecisionTreeClassifier(max_depth=5, max_features=4,
                                                   min_samples_leaf=2),
                  n_estimators=100, random_state=17)

In [28]:
b_pred_best = best_bag.predict(X_test)

In [29]:
accuracy3 = accuracy_score(y_test, b_pred_best)
accuracy3

0.8666666666666667

**Такая же проблема как и случайного леса - из-за размера датасета точность "лучшего" бэггинга может быть меньше, тем более не перебирали варианты с базовым деревом. Здесь сработало как и ожидалось**

# Итог

Бэггинг деревьев - мы берём строим 100 разных деревьев (на разных подвыборках) с одинаковым набором признаков
Случайный лес - мы берём строим 100 разных деревьев, но в них также и случайным образом меняем признаки.

По итогу - случайный лес более разнообразный по набору деревьев, чем бэггинг

В данном датасете получилось так, что бэггинг проявил себя лучше - всё исходит из той ошибки, что я говорил - маленький размер датасета (классификаторы обучаются на трейне, но на тесте могут проявлять себя по разному)

# Bias-variance декомпозиция

In [11]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.ensemble import BaggingClassifier, RandomForestClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.model_selection import train_test_split
import warnings
warnings.filterwarnings('ignore')
import os

# Создаем папку для сохранения графиков
if not os.path.exists('plots'):
    os.makedirs('plots')

# Загружаем данные
data = pd.read_csv("heart_failure_clinical_records_dataset.csv")
X = data.drop('DEATH_EVENT', axis=1)
y = data['DEATH_EVENT']

# Параметры для повторных экспериментов
n_repeat = 30
test_size = 0.2
np.random.seed(42)

estimators = [
    ("Decision Tree", DecisionTreeClassifier(random_state=42, max_depth=5)),
    ("Bagging (Tree)", BaggingClassifier(
        DecisionTreeClassifier(random_state=42, max_depth=5), 
        n_estimators=50, 
        random_state=42
    )),
    ("Random Forest", RandomForestClassifier(
        n_estimators=50, 
        max_depth=5,
        random_state=42
    ))
]

def evaluate_bias_variance_classification(estimator, X, y, n_repeat=30, test_size=0.2):
    all_predictions = []
    all_true_labels = []
    X_test_fixed = None
    y_test_fixed = None
    
    for i in range(n_repeat):
        X_train, X_test, y_train, y_test = train_test_split(
            X, y, test_size=test_size, random_state=i, stratify=y
        )
        
        estimator.fit(X_train, y_train)
        
        if hasattr(estimator, "predict_proba"):
            y_pred_proba = estimator.predict_proba(X_test)[:, 1]
        else:
            y_pred_proba = estimator.predict(X_test).astype(float)
        
        all_predictions.append(y_pred_proba)
        all_true_labels.append(y_test.values)
        
        if i == 0:
            X_test_fixed = X_test
            y_test_fixed = y_test
    
    all_predictions = np.array(all_predictions)
    mean_prediction = np.mean(all_predictions, axis=0)
    variance = np.var(all_predictions, axis=0)
    bias_squared = (mean_prediction - y_test_fixed.values) ** 2
    noise = np.full(len(y_test_fixed), 0.01)
    total_error = bias_squared + variance + noise
    
    return {
        'total_error': total_error,
        'bias_squared': bias_squared,
        'variance': variance,
        'noise': noise,
        'mean_prediction': mean_prediction,
        'X_test': X_test_fixed,
        'y_test': y_test_fixed,
        'y_test_values': y_test_fixed.values
    }

# Анализируем каждый классификатор
print("=" * 80)
print("Анализ смещения и дисперсии для классификаторов")
print("=" * 80)

results = {}
for name, estimator in estimators:
    print(f"\nОценка для {name}...")
    results[name] = evaluate_bias_variance_classification(estimator, X, y, n_repeat=n_repeat)
    print(f"  Средняя общая ошибка: {np.mean(results[name]['total_error']):.4f}")
    print(f"  Среднее смещение²: {np.mean(results[name]['bias_squared']):.4f}")
    print(f"  Средняя дисперсия: {np.mean(results[name]['variance']):.4f}")

# ============================================
# СОХРАНЕНИЕ ГРАФИКОВ ОТДЕЛЬНО
# ============================================

# ГРАФИКИ 1, 2, 3: Для каждой модели (предсказания + разложение ошибки)
for idx, (name, result) in enumerate(results.items()):
    fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(10, 10))
    fig.suptitle(f'{name}', fontsize=14, fontweight='bold')
    
    x_range = range(len(result['y_test_values']))
    
    # Верхний график: предсказания
    ax1.scatter(x_range, result['y_test_values'], 
                c='blue', label='Истинные метки', alpha=0.6, s=20, marker='o')
    ax1.plot(x_range, result['mean_prediction'], 
             'r-', label='Среднее предсказание', linewidth=2, alpha=0.8)
    
    std_dev = np.sqrt(result['variance'])
    ax1.fill_between(x_range, 
                      result['mean_prediction'] - std_dev,
                      result['mean_prediction'] + std_dev,
                      alpha=0.2, color='red', label='±1 std')
    
    ax1.set_xlabel('Образец в тестовой выборке', fontsize=10)
    ax1.set_ylabel('Вероятность смерти', fontsize=10)
    ax1.set_ylim([-0.1, 1.1])
    ax1.legend(loc='upper right', fontsize=9)
    ax1.grid(True, alpha=0.3)
    
    # Нижний график: разложение ошибки
    ax2.plot(x_range, result['total_error'], 
             'r-', label='Общая ошибка', linewidth=2)
    ax2.plot(x_range, result['bias_squared'], 
             'b-', label='Смещение²', linewidth=2)
    ax2.plot(x_range, result['variance'], 
             'g-', label='Дисперсия', linewidth=2)
    
    ax2.set_xlabel('Образец в тестовой выборке', fontsize=10)
    ax2.set_ylabel('Ошибка', fontsize=10)
    ax2.legend(loc='upper right', fontsize=9)
    ax2.grid(True, alpha=0.3)
    
    plt.tight_layout()
    safe_name = name.replace(' ', '_').replace('(', '').replace(')', '')
    plt.savefig(f'plots/{safe_name}_analysis.png', dpi=150, bbox_inches='tight')
    plt.close()
    print(f"✅ Сохранен: plots/{safe_name}_analysis.png")

# ГРАФИК 4: Сравнение метрик
fig, ax = plt.subplots(figsize=(12, 6))

x = np.arange(len(estimators))
width = 0.35

bias_values = [np.mean(results[name]['bias_squared']) for name, _ in estimators]
variance_values = [np.mean(results[name]['variance']) for name, _ in estimators]

bars1 = ax.bar(x - width/2, bias_values, width, label='Смещение²', color='blue', alpha=0.7)
bars2 = ax.bar(x + width/2, variance_values, width, label='Дисперсия', color='green', alpha=0.7)

for bars in [bars1, bars2]:
    for bar in bars:
        height = bar.get_height()
        ax.annotate(f'{height:.4f}',
                   xy=(bar.get_x() + bar.get_width() / 2, height),
                   xytext=(0, 3),
                   textcoords="offset points",
                   ha='center', va='bottom', fontsize=9)

ax.set_xlabel('Классификатор', fontsize=12)
ax.set_ylabel('Значение ошибки', fontsize=12)
ax.set_title('Сравнение смещения и дисперсии', fontsize=13, fontweight='bold')
ax.set_xticks(x)
ax.set_xticklabels([name for name, _ in estimators], fontsize=11)
ax.legend(fontsize=11)
ax.grid(True, alpha=0.3, axis='y')

plt.tight_layout()
plt.savefig('plots/metrics_comparison.png', dpi=150, bbox_inches='tight')
plt.close()
print("✅ Сохранен: plots/metrics_comparison.png")

# ГРАФИК 5: Соотношение дисперсия/смещение
fig, ax = plt.subplots(figsize=(10, 6))

ratios = [np.mean(results[name]['variance'])/np.mean(results[name]['bias_squared']) 
          for name, _ in estimators]
names = [name for name, _ in estimators]

bars = ax.bar(names, ratios, color=['orange', 'lightgreen', 'lightblue'], alpha=0.7)
ax.axhline(y=1, color='red', linestyle='--', label='Равное соотношение (1:1)', linewidth=2)

for bar, ratio in zip(bars, ratios):
    height = bar.get_height()
    ax.annotate(f'{ratio:.2f}',
               xy=(bar.get_x() + bar.get_width() / 2, height),
               xytext=(0, 3),
               textcoords="offset points",
               ha='center', va='bottom', fontsize=10)

ax.set_xlabel('Классификатор', fontsize=12)
ax.set_ylabel('Соотношение (Дисперсия / Смещение²)', fontsize=12)
ax.set_title('Соотношение дисперсии к смещению', fontsize=13, fontweight='bold')
ax.legend(fontsize=10)
ax.grid(True, alpha=0.3, axis='y')

plt.tight_layout()
plt.savefig('plots/variance_bias_ratio.png', dpi=150, bbox_inches='tight')
plt.close()
print("✅ Сохранен: plots/variance_bias_ratio.png")

# ГРАФИКИ 6, 7, 8: Только предсказания
for idx, (name, result) in enumerate(results.items()):
    fig, ax = plt.subplots(figsize=(10, 6))
    
    x_range = range(len(result['y_test_values']))
    
    ax.scatter(x_range, result['y_test_values'], 
               c='blue', label='Истинные метки', alpha=0.6, s=30, marker='o')
    ax.plot(x_range, result['mean_prediction'], 
            'r-', label='Среднее предсказание', linewidth=2, alpha=0.8)
    
    std_dev = np.sqrt(result['variance'])
    ax.fill_between(x_range, 
                     result['mean_prediction'] - std_dev,
                     result['mean_prediction'] + std_dev,
                     alpha=0.2, color='red', label='±1 std')
    
    ax.set_xlabel('Образец в тестовой выборке', fontsize=12)
    ax.set_ylabel('Вероятность смерти', fontsize=12)
    ax.set_title(f'{name}', fontsize=13, fontweight='bold')
    ax.set_ylim([-0.1, 1.1])
    ax.legend(loc='upper right', fontsize=10)
    ax.grid(True, alpha=0.3)
    
    plt.tight_layout()
    safe_name = name.replace(' ', '_').replace('(', '').replace(')', '')
    plt.savefig(f'plots/{safe_name}_predictions_only.png', dpi=150, bbox_inches='tight')
    plt.close()
    print(f"✅ Сохранен: plots/{safe_name}_predictions_only.png")

# ГРАФИКИ 9, 10, 11: Только разложение ошибки
for idx, (name, result) in enumerate(results.items()):
    fig, ax = plt.subplots(figsize=(10, 6))
    
    x_range = range(len(result['y_test_values']))
    
    ax.plot(x_range, result['total_error'], 
            'r-', label='Общая ошибка', linewidth=2)
    ax.plot(x_range, result['bias_squared'], 
            'b-', label='Смещение²', linewidth=2)
    ax.plot(x_range, result['variance'], 
            'g-', label='Дисперсия', linewidth=2)
    
    ax.set_xlabel('Образец в тестовой выборке', fontsize=12)
    ax.set_ylabel('Ошибка', fontsize=12)
    ax.set_title(f'{name}', fontsize=13, fontweight='bold')
    ax.legend(loc='upper right', fontsize=10)
    ax.grid(True, alpha=0.3)
    
    plt.tight_layout()
    safe_name = name.replace(' ', '_').replace('(', '').replace(')', '')
    plt.savefig(f'plots/{safe_name}_error_decomposition.png', dpi=150, bbox_inches='tight')
    plt.close()
    print(f"✅ Сохранен: plots/{safe_name}_error_decomposition.png")

# Итоговый вывод
print("\n" + "=" * 80)
print("ВСЕ ГРАФИКИ СОХРАНЕНЫ В ПАПКЕ 'plots'")
print("=" * 80)
print("\nСписок сохраненных файлов:")
for file in sorted(os.listdir('plots')):
    if file.endswith('.png'):
        print(f"  📊 {file}")

Анализ смещения и дисперсии для классификаторов

Оценка для Decision Tree...
  Средняя общая ошибка: 0.3954
  Среднее смещение²: 0.2110
  Средняя дисперсия: 0.1744

Оценка для Bagging (Tree)...
  Средняя общая ошибка: 0.3374
  Среднее смещение²: 0.2143
  Средняя дисперсия: 0.1131

Оценка для Random Forest...
  Средняя общая ошибка: 0.2930
  Среднее смещение²: 0.2155
  Средняя дисперсия: 0.0675
✅ Сохранен: plots/Decision_Tree_analysis.png
✅ Сохранен: plots/Bagging_Tree_analysis.png
✅ Сохранен: plots/Random_Forest_analysis.png
✅ Сохранен: plots/metrics_comparison.png
✅ Сохранен: plots/variance_bias_ratio.png
✅ Сохранен: plots/Decision_Tree_predictions_only.png
✅ Сохранен: plots/Bagging_Tree_predictions_only.png
✅ Сохранен: plots/Random_Forest_predictions_only.png
✅ Сохранен: plots/Decision_Tree_error_decomposition.png
✅ Сохранен: plots/Bagging_Tree_error_decomposition.png
✅ Сохранен: plots/Random_Forest_error_decomposition.png

ВСЕ ГРАФИКИ СОХРАНЕНЫ В ПАПКЕ 'plots'

Список сохраненных фа